<a href="https://colab.research.google.com/github/iannickgagnon/notebooks/blob/main/design_pattern_decorator_strategy_visitor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from __future__ import annotations

from random import randint, choice

from dataclasses import dataclass, asdict

from typing import Protocol, runtime_checkable

# Generic Knapsack object class

In [5]:
@dataclass
class KnapsackObject:
  """
  Generic Knapsack object class.

  Attributes:
    identifier (int): Object identifier.
    weight (int): Weight of the object.
    value (int): Value of the object.
  """
  identifier: str
  weight: int
  value: int

  def __post__init__(self):
    """
    Validate attributes after initialization.
    """
    if self.weight <= 0:
      raise ValueError("Weight must be > 0")

    if self.value < 0:
      raise ValueError("Value cannot be negative")

  def accept(self, visitor: KnapsackObjectVisitor):
    """
    Visitor pattern method for double dispatching.
    """
    raise NotImplementedError("The accept method is only implemented in the concrete subclasses.")

# Concrete KnapsckObject classes

In [6]:
class Television(KnapsackObject):
  """
  Concrete KanpsackObject class that corresponds to a television.

  Attributes:
    name (str): Name of the object.
  """
  name:str = "Television"

  def __init__(self, identifier: int, weight: int, value: int):
    """"
    Calls the parent class constructor.
    """
    super().__init__(identifier=f"{self.name} {identifier}", weight=weight, value=value)

  def accept(self, visitor: KnapsackObjectVisitor):
    """
    Double dispatching (i.e., depens on KnapsackObject and KnapsackObjectVisitor).
    """
    return visitor.visit_television(self)


class Vase(KnapsackObject):
  """
  Concrete KnapsackObject class that corresponds to a vase.
  """
  name:str  = "Vase"

  def __init__(self, identifier: int, weight: int, value: int):
    """
    Calls the parent class constructor.
    """
    super().__init__(identifier=f"{self.name} {identifier}", weight=weight, value=value)

  def accept(self, visitor: KnapsackObjectVisitor):
    """
    Double dispatching (i.e., depens on KnapsackObject and KnapsackObjectVisitor).
    """
    return visitor.visit_vase(self)

# Concrete visitors

In [7]:
@runtime_checkable
class KnapsackObjectVisitor(Protocol):
  """
  Visitor interface for KnapsackObject elements.

  Protocol defines a structural interface, which amounts to duck typing
  with type checking.

  That means:
    - A class does NOT need to inherit from `KnapsackObjectVisitor` to be
      considered a valid visitor.
    - It only needs to provide the same methods with compatible signatures.

  Example:

    class MyVisitor:
      def visit_television(self, obj: Television) -> float: ...
      def visit_vase(self, obj: Vase) -> float: ...

    `MyVisitor` is accepted anywhere a `KnapsackObjectVisitor` is expected,
    because it has the right method names/signatures.

  This is useful for the Visitor pattern because you can add new visitors
  (new operations) without modifying existing object classes, and without
  forcing visitors to inherit from a common base class.

  Normally, `Protocol` is meant for static type checkers (mypy/pyright).

  At runtime, Python does not enforce protocol conformance automatically.

  Decorating the Protocol with @runtime_checkable enables runtime checks like:

      isinstance(visitor, KnapsackObjectVisitor)

  With the decorator:
    - `isinstance` will return True if the object has the required attributes
      (here: `visit_television` and `visit_vase`) with a compatible shape.

    - It’s still not perfect "full type checking" at runtime; it’s mainly an
      structural attribute-presence check, not a guarantee about argument
      and return types.

  Without the decorator:
    - isinstance(visitor, KnapsackObjectVisitor) raises a TypeError.

  The `...` is the literal Python `Ellipsis` object.

  In a Protocol, these method bodies are stubs:
    - They say “this method must exist” and define its signature.
    - They do not provide an implementation.

  The type checker reads these stubs as abstract requirements.

  With this Protocol, the `accept()` methods can be typed as:
      def accept(self, visitor: KnapsackObjectVisitor) -> float:
          ...

  Then any new visitor class (InverseWeightVisitor, ValueDensityVisitor, etc.)
  will type-check as long as it defines the required methods with no inheritance
  required.
  """
  def visit_television(self, obj: Television) -> float: ...
  def visit_vase(self, obj: Vase) -> float: ...

class InverseWeightVisitor:
  """
  Returns the inverse of the object's weight.
  """
  def visit_television(self, obj: Television) -> float:
    return 1.0 / obj.weight

  def visit_vase(self, obj: Vase) -> float:
    return 1.0 / obj.weight

class ValueDensityVisitor:
  """
  Returns the value density (value/weight) of the object.
  """
  def visit_television(self, obj: Television) -> float:
    return obj.value / obj.weight

  def visit_vase(self, obj: Vase) -> float:
    return obj.value / obj.weight

# Data generator

In [8]:
@dataclass
class DataGenerator:
  """
  Generates a list of random objects for the Knapsack Problem.

  Attributes:
    weight_min_television_lb (int): Minimum weight of a television in pounds (lb).
    weight_max_television_lb (int): Maximum weight of a television in pounds (lb).
    value_min_television (int): Minimum value of a television in dollars.
    value_max_television (int): Maximum value of a television in dollars.
    weight_min_vase_lb (int): Minimum weight of a vase in pounds (lb).
    weight_max_vase_lb (int): Maximum weight of a vase in pounds (lb).
    value_min_vase (int): Minimum value of a vase in dollars.
    velue_max_vase (int): Maximum value of a vase in dollars.
  """
  weight_min_television_lb: int = 50
  weight_max_television_lb: int = 300
  value_min_television: int = 100
  value_max_television: int = 1500
  weight_min_vase_lb: int = 1
  weight_max_vase_lb: int = 10
  value_min_vase: int = 5
  value_max_vase: int = 3000

  def __post__init__(self):
    """
    Ensures that attributes (i.e. weights and values) are greater than zero.
    """
    accumulator: list[str] = []
    for attribute, value in asdict(self).items():
      if value <= 0:
        accumulator.append(attribute)
    if accumulator:
      raise ValueError(f"The following attributes must be greater than zero : {accumulator}")


  def generate(self, nb_objects: int = 10, verbose: bool = True) -> list[KnapsackObject]:
    """
    Generates a list of nb_objects random objects.

    Args:
      nb_objects (int, optional): The number of objects to generate. Defaults to 10.
      verbose (bool, optional): Whether to print progress or not. Defaults to True.

    Returns:
      (tuple[KnapsackObject]): A list of random KnapsackObject instances.

    Raises:
      ValueError: When the supplied number of objects is less than zero.
    """

    # Validate number of objects
    if nb_objects < 0:
      raise ValueError("The number of objects to generate must be >= 0")

    # Initialize return container
    objects: list[KnapsackObject] = []

    # Generate random objects
    for i in range(nb_objects):

      # NOTE: Type is simplified for readability
      obj_factory: tuple = choice([(Television,
                            self.weight_min_television_lb,
                            self.weight_max_television_lb,
                            self.value_min_television,
                            self.value_max_television),
                            (Vase,
                            self.weight_min_vase_lb,
                            self.weight_max_vase_lb,
                            self.value_min_vase,
                            self.value_max_vase)])

      weight_min: int = obj_factory[1]
      weight_max: int = obj_factory[2]
      value_min: int = obj_factory[3]
      value_max: int = obj_factory[4]

      random_weight: int = randint(weight_min, weight_max)
      random_value: int = randint(value_min, value_max)

      random_object: KnapsackObject = obj_factory[0](identifier=i + 1,
                                                     weight=random_weight,
                                                     value=random_value)

      objects.append(random_object)

      # Show progress
      if verbose:
        print(f"Added object no.{i + 1}: {random_object}")

    return objects

# Run generator
objects = DataGenerator().generate()

Added object no.1: Vase(identifier='Vase 1', weight=8, value=2861)
Added object no.2: Television(identifier='Television 2', weight=109, value=290)
Added object no.3: Television(identifier='Television 3', weight=111, value=1161)
Added object no.4: Television(identifier='Television 4', weight=142, value=382)
Added object no.5: Television(identifier='Television 5', weight=240, value=374)
Added object no.6: Vase(identifier='Vase 6', weight=3, value=265)
Added object no.7: Vase(identifier='Vase 7', weight=5, value=1786)
Added object no.8: Vase(identifier='Vase 8', weight=4, value=163)
Added object no.9: Vase(identifier='Vase 9', weight=8, value=890)
Added object no.10: Television(identifier='Television 10', weight=212, value=1421)


# Do something with the visitors
Calculate the inverse weights of the objects in the list.

In [9]:
inverse_weight_visitor = InverseWeightVisitor()
value_density_visitor = ValueDensityVisitor()

inverse_weights: list[float] = []
value_densities: list[float] = []
for knapsack_obj in objects:
  inverse_weights.append(knapsack_obj.accept(inverse_weight_visitor))
  value_densities.append(knapsack_obj.accept(value_density_visitor))

for weight, density in zip(inverse_weights, value_densities):
  print(f"inverse weight (1/lb) = {weight}\n"
        f"value density  ($/lb) = {density}\n")

inverse weight (1/lb) = 0.125
value density  ($/lb) = 357.625

inverse weight (1/lb) = 0.009174311926605505
value density  ($/lb) = 2.6605504587155964

inverse weight (1/lb) = 0.009009009009009009
value density  ($/lb) = 10.45945945945946

inverse weight (1/lb) = 0.007042253521126761
value density  ($/lb) = 2.6901408450704225

inverse weight (1/lb) = 0.004166666666666667
value density  ($/lb) = 1.5583333333333333

inverse weight (1/lb) = 0.3333333333333333
value density  ($/lb) = 88.33333333333333

inverse weight (1/lb) = 0.2
value density  ($/lb) = 357.2

inverse weight (1/lb) = 0.25
value density  ($/lb) = 40.75

inverse weight (1/lb) = 0.125
value density  ($/lb) = 111.25

inverse weight (1/lb) = 0.0047169811320754715
value density  ($/lb) = 6.702830188679245



# Knapsack problem instance

In [26]:
@dataclass
class KnapsackProblem:
  """
  Contains necessary data for the KnapSackProblemSolver.

  Attributes:
    objects (list[KnapsackObject]): A list of available Objects.
    weights (list[int]): The weights of the objects.
    values (list[int]): The values of the objects.
    value_densities (list[float]): The value densities (i.e. value / weight) of the objects.
  """
  objects: list[KnapsackObject]
  weights: list[int]
  values: list[int]
  value_densities: list[float]

# Knapsack problem parser

In [27]:
class KnapsackProblemParser:
  """
  Generates a KnapsackProblem instance from data.

  Attributes:
    objects (list[KnapsackObject]): Objects to choose from.
    value_densities (list[float]): Order-matching value densities of the objects.
  """

  def __init__(self,
               objects: list[KnapsackObject],
               value_densities: list[float]):

    # Store objects and inverse weights
    self.objects = objects
    self.value_densities = inverse_weights

  def parse(self) -> KnapsackProblem:
    """
    Builds KnapsackProblem object from data.

    Returns:
      (KnapsackProblem): Generated problem instance.
    """

    # Extract identifiers, weights and values as lists
    weights: list[int] = [obj.weight for obj in objects]
    values: list[int] = [obj.value for obj in objects]

    # Instanciate problem instance
    knapsack_problem = KnapsackProblem(objects=objects,
                                       weights=weights,
                                       values=values,
                                       value_densities=self.value_densities)

    return knapsack_problem

# Create KnapsackProblem instance from data and parser

KnapsackProblem(objects=[Vase(identifier='Vase 1', weight=8, value=2861), Television(identifier='Television 2', weight=109, value=290), Television(identifier='Television 3', weight=111, value=1161), Television(identifier='Television 4', weight=142, value=382), Television(identifier='Television 5', weight=240, value=374), Vase(identifier='Vase 6', weight=3, value=265), Vase(identifier='Vase 7', weight=5, value=1786), Vase(identifier='Vase 8', weight=4, value=163), Vase(identifier='Vase 9', weight=8, value=890), Television(identifier='Television 10', weight=212, value=1421)], weights=[8, 109, 111, 142, 240, 3, 5, 4, 8, 212], values=[2861, 290, 1161, 382, 374, 265, 1786, 163, 890, 1421], value_densities=[0.125, 0.009174311926605505, 0.009009009009009009, 0.007042253521126761, 0.004166666666666667, 0.3333333333333333, 0.2, 0.25, 0.125, 0.0047169811320754715])


# Implement solver strategies

In [29]:
from abc import ABC, abstractmethod

In [68]:
class Knapsack:
  """
  A simple knapsack.

  Attributes:
    capacity_lb (int): The knapsack's capacity in pounds (lb).
    content (list[KnapsackObject]): The knapsack's content. Default is an empty list.
    current_weight (int): Total weight of the knapsack's content. Defaults to zero.
    total_value (int): Total value of the knapsack's content. Defaults to zero.
  """

  def __init__(self, capacity_lb: int):
    self.capacity_lb: int = capacity_lb
    self.content: list[KnapsackObject] = []
    self.total_weight: int = 0
    self.total_value: int = 0

  def does_it_fit(self, obj: KnapsackObject):
    """
    Checks if the object fits in the knapsack given its current content
    and capabity.

    Args:
      obj (KnapsackObject): An object (e.g. Television or Vase).

    Returns:
      bool: Whether it fits or not.
    """
    return self.total_weight + obj.weight <= self.capacity_lb


  def add_item(self, obj: KnapsackObject) -> None:
    """
    Stores an item in the knapsack.

    Args:
      obj (KnapsackObject): Object to be stored.

    Raises:
      ValueError: If adding the object surpasses the knapsack's capacity.
    """

    # Make sure the item fits
    total_weight: int = self.total_weight + obj.weight
    if not self.does_it_fit(obj):
      raise ValueError(f"Total capacity exceeded: {total_weight} lbs vs. {self.capacity_lb} lbs)")

    # Put object in knapack
    self.content.append(obj)

    # Update weight and value
    self.total_weight = total_weight
    self.total_value += obj.value

In [49]:
class KnapsackSolverStrategy(ABC):
  """
  Strategy pattern for our solver.
  """
  @abstractmethod
  def solve(self, problem: KnapsackProblem) -> None:
    pass

In [50]:
class GreedyMaximumValueStrategy(KnapsackSolverStrategy):
  """
  Concrete KnapsackProblemSolver strategy.

  Add the items with the highest values first until either:
    1. All the items are in the knapsack.
    2. The knapsack is full.
  """

  def solve(self, problem: KnapsackProblem, knapsack: Knapsack) -> None:

    # Value given to one of the problem's objects when it is alredy in the knapsack
    INVALID_VALUE: int = -1

    # Copy the list of values to avoid modifying the problem instance
    values_proxy: list[int] = problem.values.copy()

    while True:

      # Find next most valuable object's value
      highest_value: int = max(values_proxy)

      # All items are in the bag
      if highest_value == INVALID_VALUE:
        break

      # Find next most valuable object's position and reference
      most_valuable_idx: int = problem.values.index(highest_value)
      most_valuable_object: KnapsackObject = problem.objects[most_valuable_idx]

      # Add if it fits, leave otherwise
      if knapsack.does_it_fit(most_valuable_object):
        knapsack.add_item(most_valuable_object)

        # Remove item from proxy (i.e. remove from bag)
        values_proxy[most_valuable_idx] = INVALID_VALUE

      else:
        break

In [60]:
# Initialize problem parser
parser = KnapsackProblemParser(objects=objects,
                               value_densities=value_densities)

# Initialize problem instance
problem_instance: KnapsackProblem = parser.parse()

# Initialize knapsack
knapsack: Knapsack = Knapsack(capacity_lb=350)

# Test strategy by itself
strategy = GreedyMaximumValueStrategy()
strategy.solve(problem=problem_instance,
               knapsack=knapsack)

In [67]:
# Show solution
print(f"Number of items : {len(knapsack.content)}")
print(f"Total value     : {knapsack.total_value}")

print("\nKnapsack content:\n")

for i, obj in enumerate(knapsack.content):
  print(f"\tObject no.{i + 1} : {obj}")

Number of items : 5
Total value     : 8119

Knapsack content:

	Object no.1 : Vase(identifier='Vase 1', weight=8, value=2861)
	Object no.2 : Vase(identifier='Vase 7', weight=5, value=1786)
	Object no.3 : Television(identifier='Television 10', weight=212, value=1421)
	Object no.4 : Television(identifier='Television 3', weight=111, value=1161)
	Object no.5 : Vase(identifier='Vase 9', weight=8, value=890)
